# Jina Reranker Pipeline for News QA System
Notebook này chạy **Jina Reranker (jinaai/jina-reranker-v2-base-multilingual)** trên kết quả đầu ra của 4 file embedding (`per_query_token.jsonl`, `per_query_structured.jsonl`, v.v.) và lấy **Top 5** context phục vụ cho RAG.

**Thông tin Task của My:**
- **File đầu vào:** `per_query_token.jsonl`
- **Mô hình Reranker:** Jina Reranker (`jinaai/jina-reranker-v2-base-multilingual`)
- **File đầu ra:** `rerank_token_jina_top5.jsonl`

Notebook này hỗ trợ cả môi trường **Google Colab**, **Kaggle** và **Local** bằng cơ chế tự động nhận diện đường dẫn (Auto-Resolve Paths).

In [ ]:
# 1. Kiểm tra môi trường GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Chạy trên CPU (Khuyến nghị bật GPU trên Kaggle/Colab để chạy nhanh hơn)")

In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip uninstall -y transformers tokenizers
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" accelerate sentence-transformers einops pandas numpy tqdm huggingface_hub

In [ ]:
# 3. Tự động phát hiện môi trường chạy và cấu hình đường dẫn
import os
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

print(f"Môi trường chạy phát hiện được: {'Kaggle' if IS_KAGGLE else 'Google Colab' if IS_COLAB else 'Local'}")

if IS_COLAB:
    from google.colab import drive
    print("Đang kết nối Google Drive...")
    drive.mount('/content/drive')
    
    input_dir = Path("/content/drive/MyDrive/out_embedding")
    output_dir = Path("/content/drive/MyDrive/out_reranker")
    repo_root = Path("/content/Text-Mining---RAG-on-News")
    
    if not repo_root.exists():
        %cd /content
        !git clone -b Alibaba https://github.com/TiiAyyLuvBear/Text-Mining---RAG-on-News.git
    %cd {repo_root}
elif IS_KAGGLE:
    input_dir = Path("/kaggle/working")
    output_dir = Path("/kaggle/working/output_Rerank")
    repo_root = Path("/kaggle/working/Text-Mining---RAG-on-News")
    
    if not repo_root.exists():
        %cd /kaggle/working
        !git clone -b Alibaba https://github.com/TiiAyyLuvBear/Text-Mining---RAG-on-News.git
    %cd {repo_root}
else:
    curr = Path.cwd()
    repo_root = curr
    for parent in [curr] + list(curr.parents):
        if (parent / "Dataset").exists():
            repo_root = parent
            break
    %cd {repo_root}
    input_dir = repo_root
    output_dir = repo_root / "src/re-ranker/output_Rerank"

output_dir.mkdir(parents=True, exist_ok=True)
print(f"\nThư mục input: {input_dir.resolve()}")
print(f"Thư mục output: {output_dir.resolve()}")
print(f"Thư mục Repo Root: {repo_root.resolve()}")

### Hugging Face Login (Yêu cầu để tải mô hình Jina Reranker v2)
Hãy tạo một HF token với quyền Read từ [HF settings](https://huggingface.co/settings/tokens), paste vào cell dưới đây để xác thực và chấp nhận điều khoản sử dụng model `jinaai/jina-reranker-v2-base-multilingual` trên Hugging Face.

In [ ]:
# 4. Đăng nhập Hugging Face
from huggingface_hub import login

# Thay hf_token bằng token của bạn
login("hf_YOUR_HF_TOKEN_HERE")

In [ ]:
# 5. Kiểm tra các file embedding và script rerank hiện có
strategies = ["token", "structured", "llamaindex", "langchain_recursive"]
input_files = {}

for strategy in strategies:
    candidates = [
        input_dir / f"per_query_{strategy}.jsonl",
        input_dir / f"src/embed/output/dense/{strategy}/per_query_{strategy}.jsonl",
        input_dir / f"output/{strategy}/per_query_{strategy}.jsonl",
        repo_root / f"per_query_{strategy}.jsonl",
        Path("/kaggle/working") / f"per_query_{strategy}.jsonl"
    ]
    for c in candidates:
        if c.exists():
            input_files[strategy] = c
            break

print("Các file embedding đầu vào tìm thấy:")
for s, p in input_files.items():
    print(f"  - Strategy '{s}': {p.resolve()}")

script_path = repo_root / "src/re-ranker/jina_rerank_token.py"
print(f"\nRerank script exists: {script_path.exists()} ({script_path.resolve()})")

In [ ]:
# 6. Chạy Jina Reranker v2 trên các file embedding đầu vào

# Lựa chọn: RUN_ALL = True sẽ chạy cả 4 file embedding, False chỉ chạy duy nhất My's task (token)
RUN_ALL = True  

my_task_strategy = "token"
strategies_to_run = list(input_files.keys()) if RUN_ALL else [my_task_strategy]

for strategy in strategies_to_run:
    if strategy not in input_files:
        print(f"Bỏ qua '{strategy}': Không tìm thấy file embedding đầu vào.")
        continue
        
    input_file = input_files[strategy]
    output_file = output_dir / f"rerank_{strategy}_jina_top5.jsonl"
    
    print(f"\n=======================================================================")
    print(f"ĐANG RERANK STRATEGY: {strategy}")
    print(f"Input: {input_file}")
    print(f"Output: {output_file}")
    print(f"=======================================================================")
    
    # Gọi script python chạy rerank
    !python src/re-ranker/jina_rerank_token.py \
        --input "{input_file}" \
        --output "{output_file}" \
        --model-name "jinaai/jina-reranker-v2-base-multilingual" \
        --top-n 5 \
        --batch-size 8 \
        --max-length 1024

In [ ]:
# 7. Kiểm tra cấu trúc file và mẫu kết quả sau khi rerank
for strategy in strategies_to_run:
    output_file = output_dir / f"rerank_{strategy}_jina_top5.jsonl"
    if output_file.exists():
        print(f"\n=== KẾT QUẢ MẪU CỦA '{strategy}' AFTER RERANK ===")
        with output_file.open("r", encoding="utf-8") as f:
            row = json.loads(f.readline())
            print("Question:", row.get("question"))
            print("Embedding Reranker:", row.get("reranker"))
            print("Reranked candidates count:", len(row.get("reranked_candidates", [])))
            print("Rerank Metrics:", row.get("rerank_metrics"))
            print("Original Retrieval Metrics:", row.get("original_retrieval_metrics"))

In [ ]:
# 8. Tổng hợp Leaderboard Reranker và lưu ra CSV
import pandas as pd

summary_rows = []
for strategy in strategies_to_run:
    output_file = output_dir / f"rerank_{strategy}_jina_top5.jsonl"
    if output_file.exists():
        rows = []
        with output_file.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line)["rerank_metrics"])
        df = pd.DataFrame(rows)
        summary = {
            "strategy": strategy,
            "reranker": "jinaai/jina-reranker-v2-base-multilingual",
            "num_queries": len(df),
            "hit@1": df["hit@1"].mean(),
            "hit@5": df["hit@5"].mean(),
            "recall@5": df["recall@5"].mean(),
            "mrr@5": df["mrr@5"].mean(),
            "ndcg@5": df["ndcg@5"].mean(),
        }
        summary_rows.append(summary)

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    summary_file = output_dir / "rerank_jina_summary.csv"
    summary_df.to_csv(summary_file, index=False, encoding="utf-8-sig")
    print(f"Saved summary to {summary_file}")
    display(summary_df)
else:
    print("Không tìm thấy file kết quả nào để tổng hợp.")

In [ ]:
# 9. Hiển thị link tải các file kết quả về máy
from IPython.display import FileLink, display

print("=== DOWNLOAD LINKS ===")
print("Click vào đường link bên dưới để tải file kết quả reranked:")
for strategy in strategies_to_run:
    output_file = output_dir / f"rerank_{strategy}_jina_top5.jsonl"
    if output_file.exists():
        try:
            rel_path = output_file.relative_to(Path.cwd())
            display(FileLink(str(rel_path)))
        except ValueError:
            print(f"{output_file.name}: {output_file.resolve()}")